# US-05-02 — Calibration Validation des six instances

Notebook préparé, **non exécuté** : aucune sortie ni seuil réel n'est inclus.
Politique figée : quantile 0,95, méthode linéaire, score > seuil.
Attacher MVTec AD `ipythonx/mvtec-ad` et les Outputs des six références indiquées par le catalogue.
Sélectionner les versions sauvegardées des notebooks ; les SHA-256 des checkpoints font autorité.
Hazelnut PatchCore : **EXP-002 uniquement**. Aucun entraînement, coreset ou accès aux images Test.

Environnement de référence PatchCore : Torch 2.11.0 / Torchvision 0.26.0,
Anomalib 2.7.0, timm 1.0.30, Lightning 2.6.6. Préparer ces dépendances avant les imports
et redémarrer la session après installation si nécessaire. Les versions réellement utilisées seront exportées.
Le clone suivant est isolé sous /kaggle/working ; il ne modifie aucun Input.

Si les dépendances manquent, exécuter dans une session Kaggle de préparation :
```python
%pip install "torch==2.11.0" "torchvision==0.26.0" "anomalib==2.7.0" "timm==1.0.30" "lightning==2.6.6"
```
Puis redémarrer le kernel et vérifier les versions ; ne pas installer ces paquets pendant une inférence en cours.


In [ ]:
from pathlib import Path
import subprocess, sys, json

CODE_COMMIT = "c708de9ce2b682ab8cea00eabd5d3e8c7b2a0ae6"
REPO = Path("/kaggle/working/industrial-defect-ai-us0502")
if not REPO.exists():
    subprocess.run(["git", "clone", "--branch", "epic/05-evaluation-benchmark",
                    "https://github.com/MRdrissi/industrial-defect-ai.git", str(REPO)], check=True)
    subprocess.run(["git", "checkout", "--detach", CODE_COMMIT], cwd=REPO, check=True)
assert subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip() == CODE_COMMIT
sys.path.insert(0, str(REPO))
from src.evaluation.run_calibration import read_catalog, run_instance
from src.evaluation.calibration import load_threshold, export_threshold_registry, load_threshold_registry
from src.evaluation.exporters import file_sha256
catalog = read_catalog()


## Inputs et récupération des checkpoints

Le catalogue fournit les URLs historiques, les noms des fichiers et les SHA-256 attendus.
L'attachement des Outputs via **Add Input** est préférable pour choisir une version archivée.
L'option DOWNLOAD_MISSING tente aussi une récupération par l'API Kaggle autorisée si le CLI et
l'authentification sont disponibles. Le CLI peut récupérer la dernière sortie : **seul un fichier dont
le SHA-256 correspond à la référence est accepté**. En cas de différence, attacher la bonne version ;
ne jamais remplacer le hash attendu pour faire passer le contrôle.

Le code recherche les fichiers de poids uniquement ; il ne charge aucune image Test.


In [ ]:
import importlib.metadata, re, shutil
import torch

DATASET_ROOT = Path("/kaggle/input/datasets/ipythonx/mvtec-ad")
# Si l'Input a un sous-répertoire, choisir la racine contenant bottle/, hazelnut/, transistor/.
DOWNLOAD_MISSING = False
CHECKPOINT_OVERRIDES = {}  # Exemple : {"cae-hazelnut": Path("/kaggle/input/.../best_model.pt")}
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"
OUTPUT_ROOT = Path("/kaggle/working/calibration-us0502")
assert DATASET_ROOT.is_dir(), "Attacher MVTec AD et renseigner sa racine."
assert not OUTPUT_ROOT.exists(), "Choisir un nouveau dossier pour ne pas écraser un run."
for name, expected in {"anomalib": "2.7.0", "timm": "1.0.30", "lightning": "2.6.6"}.items():
    assert importlib.metadata.version(name) == expected, (name, expected)
assert torch.__version__.split("+")[0] == "2.11.0", "Utiliser le runtime Torch de référence."
assert importlib.metadata.version("torchvision").split("+")[0] == "0.26.0"

checkpoints = {}
hash_cache = {}
for item in catalog["instances"]:
    candidates = ([Path(CHECKPOINT_OVERRIDES[item["id"]])] if item["id"] in CHECKPOINT_OVERRIDES
                  else list(Path("/kaggle/input").rglob(item["checkpoint_filename"])))
    matching = []
    for path in candidates:
        if path not in hash_cache:
            hash_cache[path] = file_sha256(path)
        if hash_cache[path] == item["checkpoint_sha256"]:
            matching.append(path)
    if not matching and DOWNLOAD_MISSING:
        assert shutil.which("kaggle"), "CLI Kaggle requis pour le téléchargement optionnel."
        slug = "/".join(item["kaggle_url"].split("/code/")[1].split("?")[0].split("/")[:2])
        destination = Path("/kaggle/working/checkpoint-downloads") / item["id"]
        destination.mkdir(parents=True, exist_ok=False)
        subprocess.run(["kaggle", "kernels", "output", slug, "-p", str(destination),
                        "-f", re.escape(item["checkpoint_filename"]) + "$"], check=True)
        matching = [path for path in destination.rglob(item["checkpoint_filename"])
                    if file_sha256(path) == item["checkpoint_sha256"]]
    if matching:
        checkpoints[item["id"]] = sorted(matching)[0]
    print(item["id"], item["source_experiment"], "VERIFIED" if matching else "MISSING",
          item["kaggle_url"])


## Vérification puis calibration

Le précontrôle exige chaque image du split logique Validation (42/78/43),
sans inférer le split à partir de train/good. Les instances dont le checkpoint manque
sont laissées en attente. Les erreurs d'intégrité ne sont pas ignorées.
Chaque seuil exporté est ensuite rechargé et son intégrité vérifiée sans recalcul.


In [ ]:
from datetime import datetime, timezone
session_results = []
frozen = []
for item in catalog["instances"]:
    instance = item["id"]
    if instance not in checkpoints:
        session_results.append({"instance": instance, "status": "missing_checkpoint"})
        continue
    target = OUTPUT_ROOT / instance
    args = dict(dataset_root=DATASET_ROOT, checkpoint=checkpoints[instance],
                output_dir=target, device=DEVICE)
    print(run_instance(instance, preflight=True, **args))
    result = run_instance(instance, **args)
    session_results.append(result)
    frozen.append(load_threshold(
        target, expected_integrity_sha256=result["integrity_sha256"],
        manifest_path=REPO/item["manifest"], protocol_path=REPO/catalog["protocol"],
        expected_manifest_sha256=item["manifest_sha256"],
        expected_protocol_sha256=catalog["protocol_sha256"],
        expected_count=item["expected_validation_count"], model_family=item["model_family"],
        category=item["category"], expected_checkpoint_sha256=item["checkpoint_sha256"]))
    print(result)

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
session = {"executed_at_utc": datetime.now(timezone.utc).isoformat(),
           "code_commit": CODE_COMMIT, "results": session_results}
if len(frozen) == 6:
    anchor = export_threshold_registry(OUTPUT_ROOT/"six-thresholds", frozen)
    reloaded = load_threshold_registry(OUTPUT_ROOT/"six-thresholds",
        expected_integrity_sha256=anchor, expected_protocol_sha256=catalog["protocol_sha256"])
    assert all(reloaded[(t.model_family,t.category)].threshold.hex() == t.threshold.hex() for t in frozen)
    session["registry_integrity_sha256"] = anchor
    print("Six seuils enregistrés et rechargés sans recalibration.")
else:
    print("US-05-02 reste partielle :", len(frozen), "/ 6 calibrations.")
with (OUTPUT_ROOT/"session.json").open("x", encoding="utf-8") as stream:
    json.dump(session, stream, indent=2, allow_nan=False)
print("Session SHA-256:", file_sha256(OUTPUT_ROOT/"session.json"))


## Conservation et livraison

Enregistrer une version Kaggle **avec Outputs**, copier son URL versionnée et télécharger
`calibration-us0502/`. Chaque instance contient `validation_scores.csv`,
`calibration.json`, `integrity.json`. Le registre commun n'existe que si les six exécutions réussissent.
Conserver séparément le SHA-256 de session.json et les empreintes d'intégrité affichées,
puis les versionner avec les rapports vérifiés. Les hashes n'authentifient pas eux-mêmes leur origine.

Ne pas versionner les checkpoints ni le dataset. US-05-03 chargera les valeurs enregistrées,
vérifiera checkpoint/protocole/manifeste et appliquera strictement score > seuil.
Une nouvelle session de Test ne doit appeler ni run_instance ni export_calibration.
